# Modelagem Preditiva: Classificação de Popularidade Literária (KNN)

Este notebook documenta o desenvolvimento do pipeline de Aprendizado de Máquina Supervisionado voltado à classificação da popularidade de obras literárias. O objetivo metodológico central é basear as predições exclusivamente em metadados estruturais disponíveis na fase de pré-lançamento de um livro (como autor, formato e gênero). O algoritmo escolhido para esta etapa é o *K-Nearest Neighbors* (KNN).

## 1. Aquisição, Integração de Dados e Definição do Target
Para a construção da matriz de atributos, necessitamos de duas fontes distintas de dados pré-processados:
1. O dataset original limpo (`books_clean.parquet`), contendo os metadados bibliográficos estruturais.
2. O dataset de gêneros hierarquizados (`books_pivot_mapped.parquet`), proveniente da etapa de clusterização, que condensou a alta dimensionalidade de subgêneros em 9 categorias principais.

Realizamos a consolidação das bases (*merge*) adotando a combinação de `title` (Título) e `author` (Autor) como chave, assegurando a integridade de cada instância. Em seguida, estabelecemos o *target* do modelo definindo 3 classes de popularidade fundamentadas no volume total de avaliações (`totalratings`), refletindo o comportamento estatístico de cauda longa do mercado editorial.

In [1]:
import pandas as pd
import numpy as np

# 1. Carregando o dataset de gêneros mapeados e pivotados
df_generos = pd.read_parquet('../data/processed/books_pivot_mapped.parquet')

# 2. Carregando o dataset original limpo (que contém as colunas estruturais que precisamos)
df_original = pd.read_parquet('../data/processed/books_clean.parquet')

# 3. Selecionando apenas as colunas essenciais do arquivo original para o cruzamento
colunas_originais_necessarias = ['title', 'author', 'bookformat']

# 4. Unindo os dois datasets através da combinação única de Título e Autor
df_class = pd.merge(
    df_original[colunas_originais_necessarias], 
    df_generos, 
    on=['title', 'author'], 
    how='inner'
)

# 5. Criação das 3 Classes Alvo baseadas no totalratings
def definir_tres_classes(ratings):
    if ratings >= 10000:
        return 1  # Classe 1: Alta Popularidade
    elif ratings >= 1000:
        return 2  # Classe 2: Média Popularidade
    else:
        return 3  # Classe 3: Baixa Popularidade

df_class['popularity_class'] = df_class['totalratings'].apply(definir_tres_classes)

print(f"Dataset carregado com sucesso: {df_class.shape[0]} linhas e {df_class.shape[1]} colunas.")
print("\nDistribuição original das 3 classes:")
print(df_class['popularity_class'].value_counts())

Dataset carregado com sucesso: 82006 linhas e 16 colunas.

Distribuição original das 3 classes:
popularity_class
3    61668
2    16650
1     3688
Name: count, dtype: int64


## 2. Engenharia e Seleção de Atributos
Com o problema estruturado em múltiplas classes, isolamos a variável predita ($y$) e iniciamos o tratamento dos atributos preditores ($X$).

As transformações nos dados categóricos incluem:
* **Frequência de Autoria:** Mapeamento da força de publicação do autor no catálogo.
* **Codificação de Formatos:** Isolar os 5 formatos literários majoritários e aplicar o *One-Hot Encoding*.
* **Macro-Gêneros:** Inclusão das variáveis binárias correspondentes às 9 categorias hierárquicas.

In [2]:
# Engenharia de Atributos Avançada Integrada (Apenas Variáveis Pré-Lançamento)

# Definindo o Alvo (y)
y = df_class['popularity_class']

# 1. Força do Autor (Frequência de publicações no catálogo)
autor_frequencia = df_class['author'].value_counts()
df_class['author_frequency'] = df_class['author'].map(autor_frequencia)

# 2. Formato do Livro (Agrupamento dos top 5 + One-Hot Encoding)
top_formatos = df_class['bookformat'].value_counts().index[:5]
df_class['format_grouped'] = df_class['bookformat'].apply(lambda x: x if x in top_formatos else 'Other')
df_formatos_encoded = pd.get_dummies(df_class['format_grouped'], prefix='format', drop_first=True)

# 3. Lista Exclusiva dos 9 Macro-Gêneros Hierárquicos do seu amigo
macro_generos = [
    'Artes, Lazer e Estilo de Vida', 'Fantasia e Ficção Científica', 
    'Ficção Geral e Literatura', 'História e Biografia', 
    'Infantojuvenil e Quadrinhos', 'Mistério, Thriller e Terror', 
    'Não-Ficção e Autodesenvolvimento', 'Outros', 'Romance'
]

# 4. Consolidação Definitiva dos Preditores (X)
X = pd.concat([
    df_class[['pages', 'author_frequency']], 
    df_formatos_encoded, 
    df_class[macro_generos]
], axis=1)

# Garante que todo o DataFrame seja numérico (convertendo True/False de dummies para 1/0)
X = X.astype(float)

print("=======================================================================")
print("               ATRIBUTOS OFICIAIS SELECIONADOS PARA O KNN              ")
print("=======================================================================")
print(f"Total de preditores validados: {X.shape[1]}")
print(list(X.columns))
print("=======================================================================")


               ATRIBUTOS OFICIAIS SELECIONADOS PARA O KNN              
Total de preditores validados: 16
['pages', 'author_frequency', 'format_Kindle Edition', 'format_Mass Market Paperback', 'format_Other', 'format_Paperback', 'format_ebook', 'Artes, Lazer e Estilo de Vida', 'Fantasia e Ficção Científica', 'Ficção Geral e Literatura', 'História e Biografia', 'Infantojuvenil e Quadrinhos', 'Mistério, Thriller e Terror', 'Não-Ficção e Autodesenvolvimento', 'Outros', 'Romance']


## 3. Particionamento e Balanceamento de Dados

O dataset apresenta um desbalanceamento natural severo, onde obras de baixa popularidade (Classe 3) correspondem à vasta maioria das instâncias. Alimentar o algoritmo diretamente com esta distribuição resultaria em um classificador enviesado, propenso a alocar todas as predições na classe majoritária.

Para mitigar esse efeito, a seguinte metodologia foi aplicada:
1. **Particionamento Estratificado:** Divisão em subconjuntos de Treinamento (70%) e Teste (30%), preservando a proporção original das classes.
2. **Padronização Matemática:** Aplicação do `StandardScaler` (Z-score). Etapa mandatória para o algoritmo KNN, garantindo que variáveis com escalas amplas (como o número de páginas) não dominem o cálculo espacial da distância euclidiana.
3. **Undersampling:** Aplicação do `RandomUnderSampler` estritamente no conjunto de treino, equalizando a representatividade das classes durante o aprendizado sem corromper a fidelidade do conjunto de teste real.

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.under_sampling import RandomUnderSampler

# 1. Separação em Treino (70%) e Teste (30%) estratificado
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# 2. Padronização dos dados (Z-Score)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. Aplicação do Undersampling Apenas no Treino
rus = RandomUnderSampler(random_state=42)
X_train_resampled, y_train_resampled = rus.fit_resample(X_train_scaled, y_train)

print("Estruturas atualizadas com sucesso!")
print(f"Formato final do treino balanceado: {X_train_resampled.shape}")

Estruturas atualizadas com sucesso!
Formato final do treino balanceado: (7746, 16)


## 4. Otimização de Hiperparâmetros (*Grid Search*)
Para conferir validade empírica ao modelo e evitar a adoção arbitrária de parâmetros baseados em heurística (*magic numbers*), a definição do hiperparâmetro $K$ (número de vizinhos) e do método de ponderação foi automatizada via Validação Cruzada (*5-fold Cross-Validation*).

O limite superior do espaço de busca para $K$ foi restringido cientificamente à raiz quadrada do número total de amostras do conjunto de treinamento equalizado ($\sqrt{N}$). Adicionalmente, o teste foi limitado a valores estritamente ímpares, prevenindo empates matemáticos na votação de classes do classificador.

In [4]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV

# 1. Definição matemática do teto de vizinhos baseada na raiz quadrada do treino
n_amostras = X_train_resampled.shape[0]
limite_k = int(np.sqrt(n_amostras))

# 2. Criação da lista de parâmetros testando apenas ímpares para evitar empates
valores_k = [k for k in range(3, min(limite_k, 41), 2)]

param_grid = {
    'n_neighbors': valores_k,
    'weights': ['uniform', 'distance']
}

# 3. Execução da Busca Científica com 5 dobras de validação cruzada
knn_base = KNeighborsClassifier()
grid_search = GridSearchCV(knn_base, param_grid, scoring='f1_macro', cv=5, n_jobs=-1)
grid_search.fit(X_train_resampled, y_train_resampled)

print(f"Melhor K encontrado: {grid_search.best_params_['n_neighbors']}")
print(f"Melhor ponderação encontrada: {grid_search.best_params_['weights']}")

Melhor K encontrado: 29
Melhor ponderação encontrada: uniform


## 5. Avaliação do Modelo Final

O estimador com a melhor configuração paramétrica validada na etapa anterior é extraído e submetido ao conjunto de teste (intocado até este momento). A performance global e direcional do modelo é analisada quantitativamente através das métricas de Precisão, *Recall* e *F1-Score*.

In [7]:
from sklearn.metrics import classification_report, accuracy_score

# 1. Avaliação final com o melhor modelo no conjunto de teste intocado
melhor_knn = grid_search.best_estimator_
previsoes_finais = melhor_knn.predict(X_test_scaled)
previsoes_treino = melhor_knn.predict(X_train_resampled)

print("=======================================================================")
print("               RELATÓRIO DE DESEMPENHO - TREINO (KNN)                ")
print("=======================================================================")
print(f"Acurácia Global (Treino): {accuracy_score(y_train_resampled, previsoes_treino) * 100:.2f}%\n")
print(classification_report(
    y_train_resampled,
    previsoes_treino,
    labels=[1, 2, 3],
    target_names=['Bestseller', 'Média Popularidade', 'Nicho']
))

# ── Métricas no conjunto de TESTE ────────────────────────────────────────────
print("=======================================================================")
print("               RELATÓRIO DE DESEMPENHO - TESTE (KNN)                 ")
print("=======================================================================")
print(f"Acurácia Global (Teste): {accuracy_score(y_test, previsoes_finais) * 100:.2f}%\n")
print(classification_report(
    y_test,
    previsoes_finais,
    labels=[1, 2, 3],
    target_names=['Bestseller', 'Média Popularidade', 'Nicho']
))
print("=======================================================================")

               RELATÓRIO DE DESEMPENHO - TREINO (KNN)                
Acurácia Global (Treino): 59.05%

                    precision    recall  f1-score   support

        Bestseller       0.58      0.63      0.61      2582
Média Popularidade       0.50      0.46      0.48      2582
             Nicho       0.68      0.68      0.68      2582

          accuracy                           0.59      7746
         macro avg       0.59      0.59      0.59      7746
      weighted avg       0.59      0.59      0.59      7746

               RELATÓRIO DE DESEMPENHO - TESTE (KNN)                 
Acurácia Global (Teste): 60.41%

                    precision    recall  f1-score   support

        Bestseller       0.13      0.58      0.21      1106
Média Popularidade       0.33      0.43      0.37      4995
             Nicho       0.91      0.65      0.76     18501

          accuracy                           0.60     24602
         macro avg       0.46      0.55      0.45     24602
      we

## 6. Persistência dos Artefatos (*Deploy*)

Confirmada a generalização do classificador perante dados não vistos, os componentes essenciais de produção — o modelo ajustado (`.pkl`) e os parâmetros de escalonamento estatístico (`scaler`) — são serializados. Este procedimento permite o consumo imediato da inteligência preditiva por aplicações externas, como interfaces *Web* ou *Dashboards* analíticos.

In [6]:
import joblib
import os

# Definição do caminho de destino
models_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "..", "Machine Learning", "models"))
os.makedirs(models_dir, exist_ok=True)

# Salvando os arquivos cruciais atualizados
joblib.dump(melhor_knn, os.path.join(models_dir, "knn_popularidade.pkl"))
joblib.dump(scaler, os.path.join(models_dir, "scaler_popularidade.pkl"))

print("Artefatos (.pkl) atualizados e exportados com sucesso para a pasta models!")

Artefatos (.pkl) atualizados e exportados com sucesso para a pasta models!
